# Refund Policy Agent

This agent handles customer inquiries about:
- Refund policies
- Return procedures
- Exchange requests
- Warranty claims

## Setup

In [ ]:
# Import required libraries
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

# Load configuration
from config_loader import load_config, get_model_name
config = load_config()
model_name = get_model_name(config)

## Define Agent State

In [ ]:
class AgentState(TypedDict):
    """State for refund policy agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Refund Policy Agent

In [ ]:
# System prompt for refund policy agent
REFUND_POLICY_PROMPT = """You are a helpful customer service agent specializing in REFUNDS, RETURNS, and EXCHANGES.

Your responsibilities:
- Explain refund policies clearly
- Guide customers through return procedures
- Process exchange requests
- Handle warranty claims
- Clarify eligibility requirements

Company policies (for demonstration):
- 30-day return window for most items
- Items must be unused and in original packaging
- Full refund issued within 5-7 business days
- Free return shipping for defective items
- Exchanges available for different sizes/colors
- Extended warranty available for electronics

Be empathetic, clear, and help resolve customer concerns professionally.
Always prioritize customer satisfaction while explaining policy constraints.
"""

def create_refund_policy_agent():
    """Create the refund policy agent"""
    
    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)
    
    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", REFUND_POLICY_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}
    
    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)
    
    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    
    return app

# Create the agent
refund_policy_agent = create_refund_policy_agent()
print("✅ Refund Policy Agent created successfully!")

## Test the Agent

In [ ]:
# Test conversation
from uuid import uuid4

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Test query 1
print("User: What's your refund policy?\n")
result = refund_policy_agent.invoke(
    {"messages": [HumanMessage(content="What's your refund policy?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 2 (with context)
print("User: I bought this item 2 months ago, can I still return it?\n")
result = refund_policy_agent.invoke(
    {"messages": [HumanMessage(content="I bought this item 2 months ago, can I still return it?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)

In [ ]:
# Test query 3 (follow-up)
print("User: What if the item is defective?\n")
result = refund_policy_agent.invoke(
    {"messages": [HumanMessage(content="What if the item is defective?")]},
    config
)
print(f"Agent: {result['messages'][-1].content}\n")
print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")